# Descriptive Statistics – Practical

We'll explore descriptive statistics hands-on using NumPy, Pandas, Matplotlib, and SciPy on a real-world dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

## 1. Load Dataset

We'll use the classic **Iris** dataset (built into seaborn) — 150 flower measurements across 3 species.

In [ ]:
df = sns.load_dataset('iris')
print(df.shape)
df.head()

In [ ]:
# Quick summary of data types and null values
df.info()

## 2. Measures of Central Tendency

In [ ]:
col = 'sepal_length'

mean   = df[col].mean()
median = df[col].median()
mode   = df[col].mode()[0]  # mode() returns a Series; take first value

print(f"Mean:   {mean:.4f}")
print(f"Median: {median:.4f}")
print(f"Mode:   {mode:.4f}")

In [ ]:
# Visualise where mean, median and mode fall on the distribution
plt.figure(figsize=(9, 4))
sns.histplot(df[col], bins=20, kde=True, color='steelblue')
plt.axvline(mean,   color='red',    linestyle='--', label=f'Mean ({mean:.2f})')
plt.axvline(median, color='green',  linestyle='--', label=f'Median ({median:.2f})')
plt.axvline(mode,   color='orange', linestyle='--', label=f'Mode ({mode:.2f})')
plt.title('Sepal Length Distribution')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Measures of Dispersion

In [ ]:
data = df[col]

print(f"Range:    {data.max() - data.min():.4f}")
print(f"Variance: {data.var():.4f}")     # pandas uses ddof=1 (sample variance) by default
print(f"Std Dev:  {data.std():.4f}")
q1, q3 = data.quantile(0.25), data.quantile(0.75)
print(f"IQR:      {q3 - q1:.4f}")

In [ ]:
# pandas describe() gives all of the above at once
df.describe()

## 4. Box Plot — Five-Number Summary & Outliers

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 5))

numeric_cols = df.select_dtypes(include='number').columns
for ax, col_name in zip(axes, numeric_cols):
    sns.boxplot(y=df[col_name], ax=ax, color='lightblue')
    ax.set_title(col_name.replace('_', ' ').title())

plt.suptitle('Box Plots – Iris Dataset', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 5. Shape — Skewness & Kurtosis

In [ ]:
for col_name in numeric_cols:
    skew = df[col_name].skew()
    kurt = df[col_name].kurt()   # pandas returns excess kurtosis (normal = 0)
    print(f"{col_name:<20} skewness={skew:+.3f}  excess kurtosis={kurt:+.3f}")

In [ ]:
# Compare a right-skewed variable vs a symmetric one
np.random.seed(42)
symmetric = np.random.normal(loc=0, scale=1, size=1000)
right_skewed = np.random.exponential(scale=1, size=1000)  # exponential is right-skewed

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(symmetric, bins=40, color='steelblue', edgecolor='white')
ax1.set_title(f'Symmetric  (skew={stats.skew(symmetric):.2f})')
ax2.hist(right_skewed, bins=40, color='salmon', edgecolor='white')
ax2.set_title(f'Right-Skewed  (skew={stats.skew(right_skewed):.2f})')
plt.tight_layout()
plt.show()

## 6. Covariance & Correlation

In [ ]:
# Correlation matrix
corr = df[numeric_cols].corr()
print(corr)

In [ ]:
# Heatmap makes correlations easy to spot
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Pearson Correlation Matrix – Iris')
plt.tight_layout()
plt.show()

In [ ]:
# Pair plot — scatter plots for every pair of features
sns.pairplot(df, hue='species', diag_kind='kde')
plt.suptitle('Pair Plot – Iris Dataset', y=1.01)
plt.show()

## 7. Outlier Detection Using IQR

In [ ]:
def detect_outliers_iqr(series: pd.Series) -> pd.Series:
    """Returns a boolean mask — True where a value is an outlier."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

for col_name in numeric_cols:
    mask = detect_outliers_iqr(df[col_name])
    print(f"{col_name:<20}: {mask.sum()} outlier(s)")

## 8. Grouped Statistics — Per Species

In [ ]:
# Group-wise mean and std — immediately useful for EDA before ML
df.groupby('species')[numeric_cols].agg(['mean', 'std']).round(2)

In [ ]:
# Violin plot — combines box plot with KDE
fig, axes = plt.subplots(1, 4, figsize=(15, 5))
for ax, col_name in zip(axes, numeric_cols):
    sns.violinplot(x='species', y=col_name, data=df, ax=ax, palette='Set2')
    ax.set_xlabel('')
plt.suptitle('Distribution by Species', fontsize=13)
plt.tight_layout()
plt.show()